# VitalLoop 2.0 â€” Week 1 EDA

Dataset: UCI Diabetes 130-US Hospitals (1999-2008), id=296. Goals: understand missingness, class imbalance, and leakage candidates *before* any cleaning/modeling decisions are locked in (per DATASET_ANALYSIS.md).

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 60)

df = pd.read_csv("../datasets/raw/diabetic_data.csv", low_memory=False)
print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 1. Target distribution

Binarize `readmitted` to `<30` (positive) vs everything else, per PROJECT_DESIGN.md Â§7 â€” expect ~11% positive.

In [2]:
print(df["readmitted"].value_counts())
print(df["readmitted"].value_counts(normalize=True))

y = (df["readmitted"] == "<30").astype(int)
print(f"\nPositive rate (<30 days): {y.mean():.4f}")

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

Positive rate (<30 days): 0.1116


## 2. Missingness

`ucimlrepo` already normalizes the source `"?"` marker to `NaN`. DATASET_ANALYSIS.md flags `weight` (~97% missing) and `payer_code`/`medical_specialty` (~40-50% missing) as the columns needing an explicit policy.

In [3]:
missing_rate = df.isna().mean().sort_values(ascending=False)
missing_rate = missing_rate[missing_rate > 0]
missing_rate.to_frame("missing_rate")

,missing_rate
weight,0.968585
max_glu_serum,0.947468
A1Cresult,0.832773
medical_specialty,0.490822
payer_code,0.395574
race,0.022336
diag_3,0.013983
diag_2,0.003518
diag_1,0.000206


## 3. Leakage candidates

Two known leakage traps per DATASET_ANALYSIS.md / RISK_ANALYSIS.md:

1. **Expired/hospice discharges** (`discharge_disposition_id` in {11, 13, 14, 19, 20, 21}) â€” these patients cannot be readmitted, so including them corrupts the label distribution.
2. **Duplicate patients** (`patient_nbr` appears more than once) â€” an encounter-level split would leak the same patient across train/eval.

In [4]:
LEAKAGE_DISPOSITIONS = {11, 13, 14, 19, 20, 21}

n_leakage_rows = df["discharge_disposition_id"].isin(LEAKAGE_DISPOSITIONS).sum()
print(
    f"Rows with expired/hospice discharge disposition: {n_leakage_rows} "
    f"({n_leakage_rows / len(df):.2%} of all rows)"
)

n_patients = df["patient_nbr"].nunique()
n_rows = len(df)
n_dup_patients = (df["patient_nbr"].value_counts() > 1).sum()
print(f"\nUnique patients: {n_patients} vs total encounters: {n_rows}")
print(
    f"Patients with >1 encounter: {n_dup_patients} "
    f"({n_dup_patients / n_patients:.2%} of unique patients)"
)

Rows with expired/hospice discharge disposition: 2423 (2.38% of all rows)

Unique patients: 71518 vs total encounters: 101766
Patients with >1 encounter: 16773 (23.45% of unique patients)


## 4. Summary â€” decisions carried into Week 2

- **Target**: binarize `readmitted` to `<30` vs rest; positive rate confirmed above.
- **Missingness policy**: drop `weight`; treat `payer_code`/`medical_specialty` missingness as an explicit category (per DATASET_ANALYSIS.md).
- **Leakage removal**: drop rows with `discharge_disposition_id` in {11, 13, 14, 19, 20, 21}.
- **Dedup**: keep one encounter per `patient_nbr` â€” required for a leakage-free train/eval split.

These four decisions are implemented as code (not just notes) in `ml/data/schema.py` and `ml/data/clean.py` in Week 2, with unit tests enforcing them.

## 5. Baseline model (Week 1 exit deliverable)

A deliberately quick, minimally-cleaned logistic regression to get an honest reference AUROC — not the calibrated LightGBM model that arrives in Week 3. Only the two leakage fixes above are applied (dedup + drop expired/hospice); everything else is left as a naive numeric/one-hot encoding so this stays a *baseline*, not an optimized model.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

baseline_df = df[~df["discharge_disposition_id"].isin(LEAKAGE_DISPOSITIONS)].copy()
baseline_df = baseline_df.drop_duplicates(subset="patient_nbr", keep="first")
print(f"Rows after leakage removal + dedup: {len(baseline_df)} (was {len(df)})")

target = (baseline_df["readmitted"] == "<30").astype(int)
drop_cols = ["encounter_id", "patient_nbr", "readmitted", "weight", "diag_1", "diag_2", "diag_3"]
X = baseline_df.drop(columns=drop_cols)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

preprocess = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_cols,
        ),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

# Patient-level split (not row-level) to avoid the same leakage this project exists to prevent.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, target, groups=baseline_df["patient_nbr"]))

model.fit(X.iloc[train_idx], target.iloc[train_idx])
proba = model.predict_proba(X.iloc[test_idx])[:, 1]
auroc = roc_auc_score(target.iloc[test_idx], proba)

print(f"\nBaseline logistic regression AUROC: {auroc:.4f}")
print("Published benchmark range for this task: ~0.64-0.69 (DATASET_ANALYSIS.md)")

Rows after leakage removal + dedup: 69990 (was 101766)



Baseline logistic regression AUROC: 0.6191
Published benchmark range for this task: ~0.64-0.69 (DATASET_ANALYSIS.md)
